# Reconhecimento de Gestos com Esqueleto Colorido

Este notebook utiliza a **MediaPipe Tasks API** para desenhar as conexões da mão com cores diferentes para cada dedo, criando um efeito visual premium.

### Controles:
- **'s'**: Salvar captura atual.
- **'q'**: Sair.

In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import os
from datetime import datetime

In [2]:
model_path = 'gesture_recognizer.task'
if not os.path.exists(model_path):
    print("ERRO: O modelo 'gesture_recognizer.task' não foi encontrado na pasta.")
else:
    print(f"Modelo '{model_path}' carregado com sucesso!")

Modelo 'gesture_recognizer.task' carregado com sucesso!


In [3]:
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(
    base_options=base_options,
    num_hands=2
)
recognizer = vision.GestureRecognizer.create_from_options(options)

In [4]:
def draw_colorful_landmarks(image, detection_result):
    annotated_image = np.copy(image)
    
    # Cores (BGR)
    THUMB_COLOR = (255, 0, 0)      # Azul
    INDEX_COLOR = (0, 255, 0)      # Verde
    MIDDLE_COLOR = (0, 255, 255)    # Amarelo
    RING_COLOR = (255, 100, 0)     # Laranja
    PINKY_COLOR = (255, 0, 255)    # Rosa
    PALM_COLOR = (255, 255, 255)   # Branco

    # Definição dos dedos por índices
    FINGERS = [
        {'indices': [1, 2, 3, 4], 'color': THUMB_COLOR, 'palm_connect': 0},    # Polegar
        {'indices': [5, 6, 7, 8], 'color': INDEX_COLOR, 'palm_connect': 0},    # Indicador
        {'indices': [9, 10, 11, 12], 'color': MIDDLE_COLOR, 'palm_connect': 0}, # Médio
        {'indices': [13, 14, 15, 16], 'color': RING_COLOR, 'palm_connect': 0},  # Anelar
        {'indices': [17, 18, 19, 20], 'color': PINKY_COLOR, 'palm_connect': 0}  # Mínimo
    ]

    for i, hand_landmarks in enumerate(detection_result.hand_landmarks):
        # Converter landmarks para pixels
        h, w, _ = annotated_image.shape
        pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]

        # 1. Desenhar conexões de cada dedo
        for finger in FINGERS:
            color = finger['color']
            indices = finger['indices']
            
            # Conectar base do dedo ao pulso (0)
            cv2.line(annotated_image, pts[0], pts[indices[0]], PALM_COLOR, 2)
            
            # Conectar partes do dedo
            for k in range(len(indices) - 1):
                cv2.line(annotated_image, pts[indices[k]], pts[indices[k+1]], color, 3)

        # 2. Desenhar articulações (landmarks)
        for pt in pts:
            cv2.circle(annotated_image, pt, 5, (255, 255, 255), -1)  # Borda branca
            cv2.circle(annotated_image, pt, 3, (0, 0, 0), -1)        # Centro preto

        # 3. Mostrar Gesto Reconhecido
        gesture = detection_result.gestures[i][0]
        label = f"{gesture.category_name} ({gesture.score:.2f})"
        
        # Posicionar texto acima da mão
        cv2.putText(annotated_image, label, (pts[0][0], pts[0][1] - 40), 
                    cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 2)

    return annotated_image

In [5]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame = cv2.flip(frame, 1)
    rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    result = recognizer.recognize(mp_image)
    
    if result.hand_landmarks:
        display_frame = draw_colorful_landmarks(frame, result)
    else:
        display_frame = frame.copy()
        cv2.putText(display_frame, "Mostre suas maos!", (50, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow('Colorful Gesture Tracking', display_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    if cv2.waitKey(1) & 0xFF == ord('s'):
        cv2.imwrite(f"capturas_gestos/gesto_colorido_{datetime.now().strftime('%H%M%S')}.jpg", display_frame)

cap.release()
cv2.destroyAllWindows()